# 線性代數之機器學習基礎應用

## 學習目標

完成本 Notebook 後，你將能夠：

1. 使用向量與矩陣表示機器學習中的樣本、特徵與模型參數。
2. 以 NumPy 實作點積、矩陣乘法、L2 範數與批次預測。
3. 理解線性變換如何改變特徵空間中的資料分布。
4. 使用 PCA 與 SVD 示範矩陣分解與維度簡化。
5. 使用最小平方估計求解線性迴歸參數。

本章的重點不是背公式，而是看懂「資料如何被矩陣表示，模型如何透過矩陣運算完成預測與轉換」。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節會使用到的 Python 套件，並設定圖表與亂數種子，確保每次執行結果一致。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.grid'] = True

print('環境設定完成')
print('NumPy version:', np.__version__)


## 核心概念：向量與矩陣表示

在機器學習中，一筆資料通常可表示為一個向量，多筆資料則可組成一個矩陣。

例如一筆學生資料包含三個特徵：學習時數、練習題數、模擬考分數，則可寫成：

`x = [學習時數, 練習題數, 模擬考分數]`

若模型有一組參數向量：

`theta = [theta1, theta2, theta3]`

線性模型的預測可透過點積完成：

`y_hat = theta^T x`

若一次處理多筆資料，則可使用特徵矩陣 `X` 與參數向量 `theta`：

`y_hat = X theta`

這就是許多機器學習模型能夠高效率批次運算的原因。


In [ ]:
# ── 示範：向量點積、L2 範數與矩陣批次預測 ────────────────────
# 這段程式碼示範如何用向量表示單筆樣本，用矩陣表示多筆樣本，並以點積與矩陣乘法完成線性預測。

import numpy as np
import pandas as pd

# 單筆樣本：學習時數、練習題數、模擬考分數
x = np.array([12, 80, 75])

# 模型參數：每個特徵對預測結果的權重
theta = np.array([1.5, 0.2, 0.6])

# 點積：單筆樣本的線性預測
y_hat_single = np.dot(theta, x)

# L2 範數：衡量向量長度
x_norm = np.linalg.norm(x, ord=2)

print('單筆樣本 x:', x)
print('參數 theta:', theta)
print('點積預測 y_hat:', round(y_hat_single, 2))
print('x 的 L2 範數:', round(x_norm, 2))

# 多筆樣本組成特徵矩陣 X
X = np.array([
    [12, 80, 75],
    [8, 50, 62],
    [15, 90, 88],
    [5, 30, 55]
])

# 矩陣乘法：批次預測
predictions = X @ theta

result = pd.DataFrame(X, columns=['學習時數', '練習題數', '模擬考分數'])
result['預測分數'] = np.round(predictions, 2)

print('\n批次預測結果：')
print(result)


## 核心概念：線性變換與特徵空間

線性變換可以理解為使用矩陣重新表示資料。當資料點 `x` 左乘矩陣 `A` 時：

`x_new = A x`

可能產生以下效果：

1. 縮放：放大或縮小某些方向。
2. 旋轉：改變資料點方向，但保留相對結構。
3. 剪切：讓資料分布產生傾斜。
4. 投影：將高維資料壓縮到較低維度空間。

在機器學習中，線性迴歸、邏輯迴歸、PCA、神經網路前向傳播，都可以看到線性變換的影子。


In [ ]:
# ── 示範：線性變換對資料空間的影響 ─────────────────────────
# 這段程式碼建立一組二維資料點，並套用縮放、旋轉與剪切矩陣，觀察資料在特徵空間中的幾何變化。

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.random.randn(80, 2)

scale_matrix = np.array([
    [2.0, 0.0],
    [0.0, 0.5]
])

angle = np.deg2rad(35)
rotation_matrix = np.array([
    [np.cos(angle), -np.sin(angle)],
    [np.sin(angle), np.cos(angle)]
])

shear_matrix = np.array([
    [1.0, 0.8],
    [0.0, 1.0]
])

X_scaled = X @ scale_matrix.T
X_rotated = X @ rotation_matrix.T
X_sheared = X @ shear_matrix.T

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharex=True, sharey=True)
plots = [
    ('原始資料', X),
    ('縮放', X_scaled),
    ('旋轉', X_rotated),
    ('剪切', X_sheared)
]

for ax, (title, data) in zip(axes, plots):
    ax.scatter(data[:, 0], data[:, 1], alpha=0.75)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

print('原始資料形狀:', X.shape)
print('縮放後資料形狀:', X_scaled.shape)
print('旋轉後資料形狀:', X_rotated.shape)
print('剪切後資料形狀:', X_sheared.shape)


## 核心概念：矩陣分解、降維與最小平方

高維資料常有資訊冗餘，矩陣分解可以幫助我們找出資料中較重要的方向。

常見方法包含：

1. 特徵值分解：適用於對稱方陣，常用於理解資料變異方向。
2. SVD：可用於任意矩陣，是 PCA、推薦系統與影像壓縮的重要基礎。
3. NMF：適合非負資料，可將資料拆成可加疊、較容易解釋的部件。

在線性迴歸中，最小平方估計則是另一個典型線性代數應用。它的目標是找到一組參數，讓預測值與真實值之間的平方誤差總和最小。


In [ ]:
# ── 實際應用：PCA 降維、SVD 近似與最小平方線性迴歸 ─────────────
# 這段程式碼示範三個常見應用：使用 PCA 將三維資料降成二維、使用 SVD 重建矩陣，以及用最小平方估計求解線性迴歸參數。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

np.random.seed(7)

# 一、PCA：將三個相關特徵降成兩個主成分
study_hours = np.random.normal(10, 3, 100)
practice_count = study_hours * 6 + np.random.normal(0, 8, 100)
mock_score = study_hours * 5 + practice_count * 0.4 + np.random.normal(0, 6, 100)

X = np.column_stack([study_hours, practice_count, mock_score])
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print('PCA 解釋變異比例:', np.round(pca.explained_variance_ratio_, 3))
print('兩個主成分合計保留資訊比例:', round(pca.explained_variance_ratio_.sum(), 3))

plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.75)
plt.xlabel('第一主成分')
plt.ylabel('第二主成分')
plt.title('PCA 二維投影')
plt.show()

# 二、SVD：用較少成分近似原始矩陣
A = np.array([
    [5, 4, 0, 0],
    [4, 5, 0, 0],
    [0, 0, 4, 5],
    [0, 0, 5, 4]
], dtype=float)

U, S, Vt = np.linalg.svd(A, full_matrices=False)
k = 2
A_approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

print('\n原始矩陣 A:')
print(A)
print('\n保留前 2 個奇異值的近似矩陣:')
print(np.round(A_approx, 2))

# 三、最小平方線性迴歸：y = X theta
X_reg = np.column_stack([np.ones(100), study_hours, practice_count])
y = mock_score

theta_hat = np.linalg.pinv(X_reg) @ y

print('\n使用偽逆求得的最小平方參數:')
print('截距:', round(theta_hat[0], 3))
print('學習時數權重:', round(theta_hat[1], 3))
print('練習題數權重:', round(theta_hat[2], 3))

model = LinearRegression()
model.fit(X_reg[:, 1:], y)
print('\nsklearn LinearRegression 對照：')
print('截距:', round(model.intercept_, 3))
print('權重:', np.round(model.coef_, 3))


In [ ]:
# ── 自我測驗 ────────────────────────────────────
# 請完成下方 TODO 填空，練習使用 NumPy 計算向量點積、L2 範數、矩陣批次預測與最小平方參數。

import numpy as np

# 自我測驗 1：向量點積與 L2 範數
sample = np.array([6, 40, 70])
weights = np.array([2.0, 0.3, 0.5])

# TODO: 使用 np.dot 計算 sample 與 weights 的線性預測
score = np.dot(sample, weights)

# TODO: 使用 np.linalg.norm 計算 sample 的 L2 範數
sample_norm = np.linalg.norm(sample, ord=2)

print('線性預測:', round(score, 2))
print('L2 範數:', round(sample_norm, 2))
# Expected: 線性預測: 59.0
# Expected: L2 範數: 80.87

# 自我測驗 2：矩陣批次預測
X_quiz = np.array([
    [6, 40, 70],
    [10, 65, 82],
    [4, 25, 60]
])

# TODO: 使用矩陣乘法 @ 一次計算三筆資料的預測值
batch_scores = X_quiz @ weights

print('批次預測:', np.round(batch_scores, 2))
# Expected: 批次預測: [59.  80.5 45.5]

# 自我測驗 3：最小平方估計
X_train = np.array([
    [1, 1],
    [1, 2],
    [1, 3],
    [1, 4]
], dtype=float)
y_train = np.array([3, 5, 7, 9], dtype=float)

# TODO: 使用偽逆 np.linalg.pinv 求解 theta_hat
quiz_theta_hat = np.linalg.pinv(X_train) @ y_train

print('最小平方參數:', np.round(quiz_theta_hat, 2))
# Expected: 最小平方參數: [1. 2.]
